# CerberusVision Qwen-2.5 7B LoRA Eğitim Merkezi 🚀

Bu notebook, `si_training.jsonl` veri setini kullanarak Qwen-2.5-7B-Instruct modelini Unsloth ile Google Colab'da (A100 GPU) fine-tune etmek için hazırlanmıştır.
**LÜTFEN ÇALIŞTIRMADAN ÖNCE:** Üst menüden `Çalışma Zamanı (Runtime) -> Çalışma Zamanı Türünü Değiştir -> A100 GPU` seçtiğinizden emin olun!

In [ ]:
# 1. Unsloth ve Gerekli Kütüphaneleri Kurun
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps -U xformers trl peft accelerate bitsandbytes


In [ ]:
# 2. Google Drive'a Bağlan ve Veriyi Yükle
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from datasets import Dataset

# Drive'ınızdaki CerberusVision/veriler/si_training.jsonl yolunu kendinize göre güncelleyin
data_path = "/content/drive/MyDrive/CerberusVision_Colab_Egitim_Seti/si_training.jsonl"
import os
if not os.path.exists(data_path):
    raise FileNotFoundError(f"\n❌ DOSYA BULUNAMADI: {data_path}\nLütfen 'CerberusVision_Colab_Egitim_Seti' klasörünü Drive ana dizinine yüklediğinizden emin olun!")

with open(data_path, 'r', encoding='utf-8') as f:
    df = pd.read_json(f, lines=True)

def format_prompt(row):
    return f"""<|im_start|>system
Sen bir konsimento belgesi ayristiricisin. Sadece gecerli JSON dondur.<|im_end|>
<|im_start|>user
{row['instructions']}\n\n{row['input']}<|im_end|>
<|im_start|>assistant
{row['output']}<|im_end|>"""

df['text'] = df.apply(format_prompt, axis=1)
dataset = Dataset.from_pandas(df[['text']])
print(f"Toplam {len(dataset)} egitim verisi yuklendi!")

In [ ]:
# 3. Modeli ve LoRA Adaptörünü Yükle
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Eğitimi Başlat
from trl import SFTTrainer
from trl import SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_num_proc = 2,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        packing = True,
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/CerberusVision/checkpoints",
        save_strategy = "steps",
        save_steps = 20,
        save_total_limit = 2,
    ),
)

import os
from transformers.trainer_utils import get_last_checkpoint

checkpoint_dir = "/content/drive/MyDrive/CerberusVision/checkpoints"
last_checkpoint = get_last_checkpoint(checkpoint_dir) if os.path.exists(checkpoint_dir) else None

if last_checkpoint is not None:
    print(f"\n📌 Kayıtlı checkpoint bulundu: {last_checkpoint}")
    print("Eğitim kaldığı yerden (resume) devam ediyor... Kemerleri bağla! 🚀")
    trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("\n🚀 A100 GPU ile Eğitim Sıfırdan Başlıyor...")
    trainer_stats = trainer.train()


In [ ]:
# 5. LoRA Adaptörünü Kaydet (Drive'a Geri Yükle)
model.save_pretrained("/content/drive/MyDrive/CerberusVision/models/qwen2.5-7b-cerberus-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/CerberusVision/models/qwen2.5-7b-cerberus-lora")
print("LoRA adaptoru basariyla kaydedildi!")